# 面试题：并行工具调用与 Join 怎样保证一致性？

面试回答：只读、无依赖的分支才可并行；Join 必须显式区分 required 和 optional，检查调用关联、资源版本和 deadline。晚到结果不能覆盖更新后的权威状态。下面用退款前的订单、支付与物流三分支读取，展示任何结果到达就合并为何危险。

## 真实案例

退款 Agent 并发读取订单、支付、物流；六个分支事件含 required 状态、版本和超时。

## 基线

基线只要收到任意两个分支就生成退款结论。

## 结果解读

手写 Join 要求三类关键事实齐全且版本一致。

## 失败案例

支付结果版本旧于订单快照时，不能拼接成一个看似完整的答案。

In [1]:
branches = [{'id':'J1','kind':'order','required':True,'status':'paid','version':9,'ok':True}, {'id':'J2','kind':'payment','required':True,'status':'captured','version':9,'ok':True}, {'id':'J3','kind':'shipping','required':True,'status':'not_shipped','version':9,'ok':True}, {'id':'J4','kind':'order','required':True,'status':'paid','version':10,'ok':True}, {'id':'J5','kind':'payment','required':True,'status':'captured','version':9,'ok':True}, {'id':'J6','kind':'shipping','required':True,'status':'timeout','version':10,'ok':False}]  # 构造两轮退款判断的六个并行分支结果。
print('并行分支输入:', branches)  # 输出分支类别、版本和是否成功。
print('教学说明：同一轮 Join 必须读取同一资源快照版本，真实系统还要携带 invocation id。')  # 说明版本语义。

并行分支输入: [{'id': 'J1', 'kind': 'order', 'required': True, 'status': 'paid', 'version': 9, 'ok': True}, {'id': 'J2', 'kind': 'payment', 'required': True, 'status': 'captured', 'version': 9, 'ok': True}, {'id': 'J3', 'kind': 'shipping', 'required': True, 'status': 'not_shipped', 'version': 9, 'ok': True}, {'id': 'J4', 'kind': 'order', 'required': True, 'status': 'paid', 'version': 10, 'ok': True}, {'id': 'J5', 'kind': 'payment', 'required': True, 'status': 'captured', 'version': 9, 'ok': True}, {'id': 'J6', 'kind': 'shipping', 'required': True, 'status': 'timeout', 'version': 10, 'ok': False}]
教学说明：同一轮 Join 必须读取同一资源快照版本，真实系统还要携带 invocation id。


In [2]:
first_round = branches[:3]  # 取出第一轮完整且版本一致的三条分支。
second_round = branches[3:]  # 取出第二轮存在旧支付与超时物流的三条分支。
def any_two_join(items):  # 定义只要任意两条成功就合并的错误基线。
    return '允许退款' if sum(item['ok'] for item in items) >= 2 else '等待'  # 忽略 required 缺失和版本不一致。
print('第一轮基线:', any_two_join(first_round), '，第二轮基线:', any_two_join(second_round))  # 输出基线对不完整第二轮的错误乐观结论。

第一轮基线: 允许退款 ，第二轮基线: 允许退款


In [3]:
def strict_join(items):  # 定义带 required 和版本门禁的手写 Join。
    required = [item for item in items if item['required']]  # 提取当前 Join 必须满足的分支。
    kinds = {item['kind'] for item in required if item['ok']}  # 收集成功返回的关键分支类别。
    versions = {item['version'] for item in required if item['ok']}  # 收集成功返回的快照版本集合。
    expected = {'order','payment','shipping'}  # 定义退款决策的三项必需事实。
    if kinds != expected:  # 检查是否缺少 required 分支或存在超时。
        return 'pending', {'kinds':kinds,'versions':versions}  # 返回 pending 而不是部分事实拼接。
    if len(versions) != 1:  # 检查跨服务读到的版本是否一致。
        return 'readback_required', {'kinds':kinds,'versions':versions}  # 要求重新读取一致性快照。
    return 'joined', {'kinds':kinds,'versions':versions}  # 返回可作为下一步输入的完整 Join。

In [4]:
joined_one = strict_join(first_round)  # 对第一轮完整分支执行严格 Join。
joined_two = strict_join(second_round)  # 对第二轮超时/旧版本分支执行严格 Join。
print('轮次 | Join 结论 | 中间证据')  # 输出一致性 Join 的结果表标题。
print('第一轮', joined_one)  # 输出完整同版本分支的合并证据。
print('第二轮', joined_two)  # 输出缺失分支和版本差异导致的待处理证据。
print('第一轮可进入退款策略，第二轮必须等待或回读，不能由两个成功分支替代第三个 required 分支。')  # 解读 Join 门禁。

轮次 | Join 结论 | 中间证据
第一轮 ('joined', {'kinds': {'order', 'payment', 'shipping'}, 'versions': {9}})
第二轮 ('pending', {'kinds': {'order', 'payment'}, 'versions': {9, 10}})
第一轮可进入退款策略，第二轮必须等待或回读，不能由两个成功分支替代第三个 required 分支。


In [5]:
wrong = any_two_join(second_round)  # 对不一致分支应用错误的任意两个结果合并。
fixed = strict_join(second_round)[0]  # 对同一分支集合应用 required/版本门禁。
print('失败案例第二轮：基线=', wrong, '，修正=', fixed)  # 展示并行结果不能只按数量合并。
print('生产差距：需要 structured concurrency、取消传播、deadline、分支 trace、幂等读与每类工具的版本语义。')  # 说明并发编排的生产能力。

失败案例第二轮：基线= 允许退款 ，修正= pending
生产差距：需要 structured concurrency、取消传播、deadline、分支 trace、幂等读与每类工具的版本语义。


In [6]:
assert joined_one[0] == 'joined'  # 验证同版本且三分支齐全时才能成功 Join。
assert joined_two[0] == 'pending'  # 验证 required 物流超时时不生成退款结论。
assert wrong == '允许退款'  # 验证数量型基线确实会对第二轮误判。